[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/mlsysbook-vi-notebooks/blob/main/vol1-ch03-ml-workflow/01-thue-lap.ipynb)

# Thuế lặp: tốc độ lặp mua được gì, và không mua được gì

Chương 3 kể vòng đời của một hệ thống học máy qua một hệ thống sàng lọc bệnh võng mạc do đái tháo
đường cho các phòng khám vùng xa. Ở đầu chương có hai ý về thời gian. Ý thứ nhất là thuế lặp
(iteration tax): một chu kỳ huấn luyện dài làm nhóm thử được ít ý tưởng hơn, nên sách kết luận rằng
tốc độ lặp (iteration velocity) là một tính năng. Ý thứ hai là ngày 153: một mô hình chính xác bị bỏ
vì ràng buộc bộ nhớ của thiết bị chỉ được phát hiện lúc triển khai. Sổ tay này tính lại cả hai, đo một
cuộc đua tìm siêu tham số (hyperparameter) thật trên MNIST, rồi xem nhiễu giữa các lần chạy lớn cỡ nào.

**Bạn sẽ làm:**
1. tính lại bài thuế lặp của chương, vẽ hai quỹ đạo và tìm tuần mô hình nhẹ vượt lên;
2. quét hai giả định của bài tính để thấy kết luận nằm sát mép tới đâu;
3. đo một cuộc đua trên MNIST: hai đội, cùng một ngân sách thời gian, một đội huấn luyện ensemble lớn,
   một đội huấn luyện mô hình nhẹ;
4. đặt hai mô hình của cuộc đua vào một đích triển khai (deployment target) 256 KB, rồi tính quy tắc
   $2^{N-1}$ cho một ràng buộc phát hiện muộn;
5. đo độ dao động giữa các hạt ngẫu nhiên (random seed), đặt cạnh chênh lệch 0.5 % mà chương nói có
   thể quyết định việc đạt ngưỡng lâm sàng.

In [ ]:
#@title Chuẩn bị: thư viện, hạt giống và hai hàm đối chiếu với sách { display-mode: "form" }
# Chạy ô này trước. Nó không cài thêm gì: mọi thư viện dùng ở đây đều có sẵn trên Colab.
import math, os, random, re, sys, time

import matplotlib.pyplot as plt
import numpy as np
import torch

# Số luồng CPU. 0 để thư viện tự chọn (trên Colab là 2).
NB_THREADS = int(os.environ.get("NB_THREADS", "0"))
if NB_THREADS:
    torch.set_num_threads(NB_THREADS)
    try:
        torch.set_num_interop_threads(NB_THREADS)
    except RuntimeError:  # chỉ đặt được một lần mỗi phiên
        pass
    from threadpoolctl import threadpool_limits
    threadpool_limits(NB_THREADS)

SEED = 42


def dat_hat_giong(seed=SEED):
    """Cùng hạt giống thì cùng kết quả, mỗi lần chạy lại."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


dat_hat_giong()
DATA = os.environ.get("NB_DATA", "data")  # nơi tải dữ liệu về
NB_STRICT = os.environ.get("NB_STRICT") == "1"
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3})

# Đọc một con số theo cách bảng số liệu của bài học viết nó (dấu chấm hay dấu phẩy).
import math
import re

_SUP = str.maketrans("⁰¹²³⁴⁵⁶⁷⁸⁹⁻⁺", "0123456789-+")
_MULT = {"tỷ": 1e9, "tỉ": 1e9, "triệu": 1e6, "nghìn": 1e3, "ngàn": 1e3,
         "K": 1e3, "M": 1e6, "B": 1e9}
_NUM = re.compile(
    r"(?<![\w.,])(-?)(\d+(?:[.,]\d+)*)"                      # sign, digits with separators
    r"(?:\s*[×x·]\s*10\s*(?:\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))|[eE]([-+]?\d+)"
    r"|\^\{?\s*([-+]?\d+)\s*\}?|([⁻⁺]?[⁰¹²³⁴⁵⁶⁷⁸⁹]+))?"                  # or a bare power, 10^{-5}
    r"(\s*%|\s*(?:tỷ|tỉ|triệu|nghìn|ngàn)(?!\w)|\s?[KMB](?![A-Za-z]))?")


def norm(text):
    """The outline's markup and typography reduced to plain text, as both sides compare it."""
    text = text.replace("{,}", ",").replace("**", "").replace("`", "")
    text = re.sub(r"[   ]", " ", text)
    text = re.sub(r"−(?=\d)", "-", text)
    text = re.sub(r"[~≈]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def _readings(digits):
    """(value, decimals) for each way the separators in `digits` can be read."""
    out = []
    groups_ok = lambda parts: all(len(p) == 3 for p in parts[1:])
    for thou, dec in ((",", "."), (".", ",")):
        if thou in digits and dec in digits:
            head, _, tail = digits.rpartition(dec)
            if dec in head or not tail.isdigit() or not groups_ok(head.split(thou)):
                continue
            out.append((float(head.replace(thou, "") + "." + tail), len(tail)))
    for sep in (",", "."):
        other = "." if sep == "," else ","
        if sep in digits and other not in digits:
            parts = digits.split(sep)
            if groups_ok(parts):
                out.append((float("".join(parts)), 0))                    # thousands
            if len(parts) == 2:
                out.append((float(parts[0] + "." + parts[1]), len(parts[1])))  # decimal
    if "." not in digits and "," not in digits:
        out.append((float(digits), 0))
    return out


def candidates(text):
    """Every (value, half_unit) the numbers in `text` can mean.

    half_unit is half of the last printed digit, scaled like the value, so `437,5 KB`
    accepts 437.544. Each number is offered as written and, when a multiplier or `%`
    follows it, scaled by that too, so `8,5 tỉ` cites 8.5 or 8.5e9 and `20 %` cites 20 or 0.2.
    """
    out = []
    for m in _NUM.finditer(norm(text)):
        sign, digits, exp_caret, exp_sup, exp_e, pow_caret, pow_sup, suffix = m.groups()
        exp = exp_caret or (exp_sup.translate(_SUP) if exp_sup else None) or exp_e
        power = pow_caret or (pow_sup.translate(_SUP) if pow_sup else None)
        scale10 = 10.0 ** int(exp) if exp else 1.0
        suffix = (suffix or "").strip()
        extra = 0.01 if suffix == "%" else _MULT.get(suffix)
        for value, decimals in _readings(digits):
            if power:
                # 10^{-5}, 2³²: an exact value, so it is matched to 1 part in 10⁹; a looser
                # comparison needs an explicit tol.
                v = value ** int(power)
                half = abs(v) * 1e-9
            else:
                v = value * scale10
                half = 0.5 * 10.0 ** -decimals * scale10
            v = -v if sign else v
            out.append((v, half))
            if extra:
                out.append((v * extra, half * extra))
    return out


def match(value, text):
    """The (value, half_unit) reading of `text` that `value` equals, or None."""
    for v, half in candidates(text):
        if math.isclose(value, v, rel_tol=1e-9, abs_tol=1e-300):
            return v, half
    return None


def well_anchored(text):
    """False for a citation too bare to identify one cell, like "1" or "2"."""
    t = norm(text)
    rest = re.sub(r"[\d.,\s\-+]", "", t)
    if rest:
        return True
    sig = re.sub(r"[^\d]", "", t).lstrip("0")
    return len(sig) >= 2


def _so(x):
    """Một con số để in: số nguyên có dấu phẩy ngăn nghìn, số thực giữ 6 chữ số có nghĩa."""
    if isinstance(x, (int, np.integer)) or (isinstance(x, float) and x.is_integer() and abs(x) < 1e15):
        return f"{int(x):,}"
    return f"{x:,.6g}"


def sach(gia_tri, nguon=None, trich=None, tol=None):
    """Một con số của sách, kèm đúng chữ của nó: trong bảng số liệu của bài học (nguon)
    hoặc nguyên văn trong chương (trich)."""
    chu = nguon if nguon is not None else trich
    if match(gia_tri, chu) is None:
        raise ValueError(f"{gia_tri!r} không phải con số ghi trong {chu!r}")
    return gia_tri


def theo_sach(ten, gia_tri, sach, nguon=None, trich=None, tol=None):
    """So một con số tính được với con số của sách. In ✓ khi khớp, ✗ kèm độ lệch khi không."""
    chu = nguon if nguon is not None else trich
    khop = match(sach, chu)
    if khop is None:
        raise ValueError(f"{sach!r} không phải con số ghi trong {chu!r}")
    sai_so = tol if tol is not None else khop[1]
    dung = abs(gia_tri - sach) <= sai_so
    print(f"{'✓' if dung else '✗'} {ten}: tính được {_so(gia_tri)} · sách: {chu}")
    if not dung:
        loi = f"lệch {_so(gia_tri - sach)}, quá sai số cho phép {_so(sai_so)}"
        if NB_STRICT:
            raise AssertionError(f"{ten}: {loi}")
        print(f"   {loi}. Nếu bạn vừa đổi tham số ở trên thì đây là điều được chờ đợi.")
    # None, not the bool: a cell ending in theo_sach(...) would otherwise echo True under the ✓.


def do_tren_may(ten, gia_tri, don_vi=""):
    """Một con số ĐO trên máy đang chạy: mỗi máy một khác, không phải con số của sách."""
    print(f"⏱ {ten}: {_so(gia_tri)} {don_vi} (đo trên máy này)".replace("  ", " "))


print(f"Python {sys.version.split()[0]} · PyTorch {torch.__version__} · NumPy {np.__version__}"
      f" · {torch.get_num_threads()} luồng CPU")

In [ ]:
import torch.nn.functional as F
import torchvision
from matplotlib.patches import Patch

MAU = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#8a5cc2"]  # thứ tự màu cố định, như các sổ tay khác
torch.use_deterministic_algorithms(True)
print("sẵn sàng")

## 1. Bài tính thuế lặp của chương

Sách so hai lựa chọn cho hệ thống sàng lọc, trong cùng một cửa sổ 26 tuần:

* một ensemble lớn huấn luyện trên ảnh đáy mắt độ phân giải cao: mỗi lần huấn luyện mất 1 tuần, xuất
  phát ở 95 %, mỗi vòng thêm một lượng giả định không đổi là 0.15 điểm phần trăm;
* một mô hình nhẹ hợp để triển khai ở biên: mỗi lần mất 1 giờ, xuất phát ở 90 %, mỗi vòng thêm 0.1
  điểm phần trăm.

Với chu kỳ 1 giờ, thời gian máy không còn là ràng buộc. Sách chỉ đếm 100 thí nghiệm hiệu quả, số thí
nghiệm một nhóm kịp thiết kế, chạy và rút ra bài học trong sáu tháng. Kết quả bị chặn bởi một trần
99 %.

**Dự đoán:** sau 26 tuần, mô hình nào cao hơn, và cao hơn bao nhiêu điểm?

In [ ]:
TUAN = sach(26, nguon="26 tuần")
GIO_MOI_TUAN = sach(168, trich="168 h/week")
TUAN_MOI_VONG_LON = sach(1, nguon="1 tuần vs 1 giờ")  # mô hình lớn: 1 tuần một lần huấn luyện
GIO_MOI_VONG_NHO = sach(1, nguon="1 tuần vs 1 giờ")   # mô hình nhẹ: 1 giờ
DAU_LON = sach(95, trich="training time: 1 week, accuracy: 95 percent")
DAU_NHO = sach(90, trich="training time: 1 hour, accuracy: 90 percent")
TANG_LON = sach(0.15, nguon="0,15 vs 0,1 điểm phần trăm")
TANG_NHO = sach(0.1, nguon="0,15 vs 0,1 điểm phần trăm")
HIEU_QUA = sach(100, trich="only 100 of them as effective experiments")
TRAN = sach(99, trich="the 99 percent ceiling")

vong_lon = TUAN / TUAN_MOI_VONG_LON
vong_co_the = TUAN * GIO_MOI_TUAN / GIO_MOI_VONG_NHO
theo_sach("số vòng của mô hình lớn", vong_lon, sach=26, trich="after 26 slower iterations")
theo_sach("số thí nghiệm mô hình nhẹ có thể chạy", vong_co_the, sach=4368, trich="4,368 possible experiments")
theo_sach("khoảng cách lúc xuất phát (điểm)", DAU_LON - DAU_NHO, sach=5,
          trich="a lightweight model starting 5 percentage points behind")

lon = DAU_LON + vong_lon * TANG_LON
nho_chua_tran = DAU_NHO + HIEU_QUA * TANG_NHO
nho = min(TRAN, nho_chua_tran)
theo_sach("mô hình lớn sau 26 tuần (%)", lon, sach=98.9, trich="reaches 98.9 percent after 26 slower iterations")
theo_sach("mô hình nhẹ, trước khi áp trần (%)", nho_chua_tran, sach=100,
          trich="reaches 100 percent after 100 effective iterations")
theo_sach("mô hình nhẹ, sau khi áp trần (%)", nho, sach=99, trich="It therefore renders as 99 percent")
print(f"mô hình nhẹ hơn mô hình lớn {nho - lon:.1f} điểm, đúng bằng {(nho - lon) / TANG_NHO:.0f} vòng của chính nó")
print(f"đội nhẹ dùng {HIEU_QUA} trong {vong_co_the:,.0f} lần chạy có thể, mức sử dụng (utilization) máy "
      f"{100 * HIEU_QUA / vong_co_the:.1f} %")

Bài tính không nói mô hình nhẹ vượt lên ở tuần nào. Nếu 100 thí nghiệm hiệu quả rải đều trên 26 tuần,
và mô hình lớn xong một vòng mỗi cuối tuần, ta tính được điều đó từ chính các giả định của sách.

In [ ]:
tuan = np.linspace(0, TUAN, 26 * 400 + 1)
quy_dao_lon = DAU_LON + TANG_LON * np.floor(tuan / TUAN_MOI_VONG_LON + 1e-9)
quy_dao_nho = np.minimum(TRAN, DAU_NHO + TANG_NHO * np.floor(tuan * HIEU_QUA / TUAN + 1e-9))
vuot = tuan[np.argmax(quy_dao_nho > quy_dao_lon)]
cham_tran = tuan[np.argmax(quy_dao_nho >= TRAN - 1e-9)]
assert quy_dao_nho[-1] > quy_dao_lon[-1] and np.all(quy_dao_nho[tuan >= vuot] > quy_dao_lon[tuan >= vuot])
print(f"mô hình nhẹ vượt mô hình lớn ở tuần {vuot:.1f}, chạm trần {TRAN} % ở tuần {cham_tran:.1f}")

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.step(tuan, quy_dao_lon, where="post", color=MAU[0], lw=2, label="mô hình lớn: 1 tuần mỗi vòng, +0.15 điểm")
ax.step(tuan, quy_dao_nho, where="post", color=MAU[1], lw=2, label="mô hình nhẹ: 100 vòng hiệu quả, +0.1 điểm")
ax.axhline(TRAN, color="0.4", ls="--", lw=1, label=f"trần {TRAN} % của bài tính")
ax.plot([vuot], [quy_dao_nho[np.argmax(tuan >= vuot)]], "o", color="0.2", ms=7)
ax.annotate(f"vượt lên ở tuần {vuot:.1f}\n(tính từ giả định của sách)", (vuot, quy_dao_nho[np.argmax(tuan >= vuot)]),
            xytext=(7, 98.3), fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.annotate(f"{lon:.1f} % (sách)", (TUAN, lon), xytext=(5, -12), textcoords="offset points", fontsize=8, color=MAU[0])
ax.annotate(f"{nho:.0f} % (sách)", (TUAN, nho), xytext=(5, 5), textcoords="offset points", fontsize=8, color=MAU[1])
ax.set_xlim(0, TUAN + 3)
ax.set_ylim(89, 100)
ax.set_xlabel("tuần")
ax.set_ylabel("độ chính xác theo mô hình (%)")
ax.set_title("Hai quỹ đạo của bài thuế lặp")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

## 2. Kết luận nằm sát mép tới đâu

Mô hình nhẹ thắng 0.1 điểm. Đó cũng là mức nó thắng **lớn nhất** có thể, vì trần 99 % chặn nó lại còn
mô hình lớn đã ở 98.9 %. Sách tự nói rằng riêng tốc độ không bảo đảm mô hình nhỏ thắng. Ô dưới đây giữ
mô hình lớn như trong sách, rồi đổi hai giả định của mô hình nhẹ: số thí nghiệm hiệu quả và mức tăng
mỗi vòng.

**Dự đoán:** nếu nhóm chỉ kịp 80 thí nghiệm hiệu quả thay vì 100, mô hình nào thắng?

In [ ]:
def chenh(so_vong, tang):
    """Mô hình nhẹ trừ mô hình lớn, theo điểm phần trăm, sau 26 tuần."""
    return np.minimum(TRAN, DAU_NHO + so_vong * tang) - lon


can_vong = math.ceil(round((lon - DAU_NHO) / TANG_NHO, 9))
can_tang = (lon - DAU_NHO) / HIEU_QUA
print(f"với +{TANG_NHO} điểm mỗi vòng, mô hình nhẹ cần ít nhất {can_vong} vòng hiệu quả để vượt")
print(f"với {HIEU_QUA} vòng, nó cần ít nhất +{can_tang:.3f} điểm mỗi vòng")
print(f"thắng nhiều nhất: {TRAN - lon:.1f} điểm · thua nhiều nhất (không vòng nào có ích): {lon - DAU_NHO:.1f} điểm")

so_vong = np.arange(0, 301)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
for tang, mau in [(0.05, MAU[2]), (TANG_NHO, MAU[1]), (0.15, MAU[4])]:
    nhan = f"+{tang} điểm mỗi vòng" + (" (sách)" if tang == TANG_NHO else "")
    ax1.plot(so_vong, chenh(so_vong, tang), color=mau, lw=2, label=nhan)
ax1.axhline(0, color="0.3", lw=1)
ax1.text(300, 0.45, f"trần {TRAN} %: thắng nhiều nhất +{TRAN - lon:.1f}", fontsize=8, ha="right", color="0.3")
ax1.plot([HIEU_QUA], [chenh(HIEU_QUA, TANG_NHO)], "o", color="0.2", ms=7)
ax1.annotate(f"sách: {HIEU_QUA} vòng, +{nho - lon:.1f} điểm", (HIEU_QUA, nho - lon), xytext=(130, -3),
             fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax1.set_xlabel("số thí nghiệm hiệu quả của mô hình nhẹ")
ax1.set_ylabel("mô hình nhẹ trừ mô hình lớn (điểm)")
ax1.set_title("Chênh lệch sau 26 tuần")
ax1.set_ylim(-9.5, 1.5)
ax1.legend(loc="lower right", fontsize=8)

n_luoi = np.linspace(20, 300, 300)
bien = (lon - DAU_NHO) / n_luoi
ax2.fill_between(n_luoi, bien, 0.3, color=MAU[1], alpha=0.15, lw=0)
ax2.fill_between(n_luoi, 0, bien, color=MAU[0], alpha=0.15, lw=0)
ax2.plot(n_luoi, bien, color="0.3", lw=1.5)
ax2.plot([HIEU_QUA], [TANG_NHO], "o", color="0.2", ms=7)
ax2.annotate("điểm của sách", (HIEU_QUA, TANG_NHO), xytext=(150, 0.16), fontsize=8,
             arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax2.text(210, 0.24, "mô hình nhẹ thắng\n(nhiều nhất 0.1 điểm)", fontsize=8, ha="center")
ax2.text(210, 0.012, "mô hình lớn thắng", fontsize=8, ha="center")
ax2.set_xlim(20, 300)
ax2.set_ylim(0, 0.3)
ax2.set_xlabel("số thí nghiệm hiệu quả của mô hình nhẹ")
ax2.set_ylabel("mức tăng mỗi vòng (điểm)")
ax2.set_title("Đường hoà: số vòng × mức tăng = 8.9 điểm")
plt.tight_layout()
plt.show()

Hai điều đọc được từ đồ thị:

* Điểm của sách nằm ngay trên đường hoà. Chỉ cần 89 thay vì 100 thí nghiệm hiệu quả, hay +0.089 thay
  vì +0.1 điểm mỗi vòng, là kết luận đổi chiều.
* Phần thắng và phần thua không cân nhau. Trần 99 % chặn phần thắng ở 0.1 điểm, còn nếu các vòng lặp
  không đem lại gì, mô hình nhẹ thua tới 8.9 điểm.

Vì vậy câu kết của sách được viết có điều kiện: tốc độ lặp có giá trị khi các ứng viên có chất lượng
đạt được tương đương và các thí nghiệm có ích có giá trị kỳ vọng giống nhau. Phần tiếp theo đo xem một
thí nghiệm thật đem lại bao nhiêu.

## 3. Một cuộc đua thật: cùng ngân sách thời gian

Hai đội cùng tìm siêu tham số cho một bộ phân loại chữ số viết tay MNIST, trên 10,000 ảnh huấn luyện,
chấm điểm trên 2,000 ảnh kiểm định (validation) riêng. Mọi con số ở phần này là **giả định** của sổ tay:

* **đội lớn** huấn luyện một ensemble 3 perceptron nhiều tầng (multilayer perceptron, MLP) 784-256-10
  trên ảnh 28×28, mỗi mạng 2 epoch, rồi lấy trung bình xác suất của ba mạng;
* **đội nhẹ** huấn luyện một MLP trên ảnh thu nhỏ 14×14, rộng 16, 32 hoặc 64, trong 1 hoặc 2 epoch.

Cả hai rút cấu hình ngẫu nhiên: tốc độ học (learning rate) của Adam từ $10^{-4}$ tới $10^{-1.5}$ theo
thang log, kích thước batch (batch size) 32, 64, 128 hoặc 256. Ngân sách là thời gian đội lớn chạy 3
thí nghiệm. Đội nhẹ chạy cho tới khi tiêu hết đúng chừng ấy thời gian; thí nghiệm cuối có thể vượt một
chút.

**Dự đoán:** đội nhẹ chạy được gấp bao nhiêu lần số thí nghiệm của đội lớn? Tỉ lệ đó lớn hơn hay nhỏ hơn
tỉ lệ số phép tính của hai loại thí nghiệm?

In [ ]:
tap = torchvision.datasets.MNIST(DATA, train=True, download=True)
anh = (tap.data.float() / 255 - 0.1307) / 0.3081
N_HL, N_KD = 10_000, 2_000
y_hl, y_kd = tap.targets[:N_HL], tap.targets[50_000:50_000 + N_KD]
X28_hl, X28_kd = anh[:N_HL].flatten(1), anh[50_000:50_000 + N_KD].flatten(1)
X14_hl = F.avg_pool2d(anh[:N_HL, None], 2).flatten(1)
X14_kd = F.avg_pool2d(anh[50_000:50_000 + N_KD, None], 2).flatten(1)
THANH_VIEN, AN_LON, EPOCH_LON = 3, 256, 2


def mlp(vao, an):
    return torch.nn.Sequential(torch.nn.Linear(vao, an), torch.nn.ReLU(), torch.nn.Linear(an, 10))


def tham_so_mlp(vao, an):
    return vao * an + an + an * 10 + 10


def huan_luyen(X, y, an, lr, batch, epoch, hat):
    torch.manual_seed(hat)
    mang = mlp(X.shape[1], an)
    toi_uu = torch.optim.Adam(mang.parameters(), lr=lr)
    g = torch.Generator().manual_seed(hat)
    for _ in range(epoch):
        thu_tu = torch.randperm(len(X), generator=g)
        for i in range(0, len(X), batch):
            c = thu_tu[i:i + batch]
            mat_mat = F.cross_entropy(mang(X[c]), y[c])
            toi_uu.zero_grad()
            mat_mat.backward()
            toi_uu.step()
    return mang.eval()


def thi_nghiem_lon(cfg, hat=0):
    mang = [huan_luyen(X28_hl, y_hl, AN_LON, cfg["lr"], cfg["batch"], EPOCH_LON, hat + j) for j in range(THANH_VIEN)]
    with torch.no_grad():
        xac_suat = sum(F.softmax(m(X28_kd), 1) for m in mang)
    return 100 * (xac_suat.argmax(1) == y_kd).float().mean().item()


def thi_nghiem_nhe(cfg, hat=0):
    mang = huan_luyen(X14_hl, y_hl, cfg["an"], cfg["lr"], cfg["batch"], cfg["epoch"], hat)
    with torch.no_grad():
        return 100 * (mang(X14_kd).argmax(1) == y_kd).float().mean().item()


rng = np.random.default_rng(SEED)


def rut_cau_hinh(nhe):
    cfg = {"lr": float(10 ** rng.uniform(-4, -1.5)), "batch": int(rng.choice([32, 64, 128, 256]))}
    if nhe:
        cfg.update(an=int(rng.choice([16, 32, 64])), epoch=int(rng.choice([1, 2])))
    return cfg


hang_lon = [rut_cau_hinh(False) for _ in range(3)]
hang_nhe = [rut_cau_hinh(True) for _ in range(2000)]
# Số phép tính huấn luyện ≈ 6 × tham số × số ảnh × số epoch (2 cho truyền xuôi, 4 cho truyền ngược).
flop_lon = 6 * THANH_VIEN * tham_so_mlp(784, AN_LON) * N_HL * EPOCH_LON
flop_nhe = 6 * np.mean([tham_so_mlp(196, a) for a in (16, 32, 64)]) * N_HL * np.mean([1, 2])
print(f"{len(y_hl):,} ảnh huấn luyện, {len(y_kd):,} ảnh kiểm định")
print(f"một thí nghiệm lớn ≈ {flop_lon / 1e9:.1f} GFLOP, một thí nghiệm nhẹ trung bình ≈ {flop_nhe / 1e9:.2f} GFLOP: "
      f"gấp {flop_lon / flop_nhe:.0f} lần")

In [ ]:
nhat_ky = {"lớn": [], "nhẹ": []}  # (giây cộng dồn, độ chính xác, cấu hình)
cong_don = 0.0
for cfg in hang_lon:
    t0 = time.perf_counter()
    dc = thi_nghiem_lon(cfg)
    cong_don += time.perf_counter() - t0
    nhat_ky["lớn"].append((cong_don, dc, cfg))
NGAN_SACH_S = cong_don

cong_don = 0.0
for cfg in hang_nhe:
    if cong_don >= NGAN_SACH_S:
        break
    t0 = time.perf_counter()
    dc = thi_nghiem_nhe(cfg)
    cong_don += time.perf_counter() - t0
    nhat_ky["nhẹ"].append((cong_don, dc, cfg))

so_lon, so_nhe = len(nhat_ky["lớn"]), len(nhat_ky["nhẹ"])
tot_lon = max(d for _, d, _ in nhat_ky["lớn"])
tot_nhe = max(d for _, d, _ in nhat_ky["nhẹ"])
do_tren_may("ngân sách: thời gian 3 thí nghiệm của đội lớn", NGAN_SACH_S, "s")
do_tren_may("số thí nghiệm đội nhẹ chạy trong cùng ngân sách", so_nhe)
do_tren_may("số thí nghiệm nhiều gấp", so_nhe / so_lon, "lần")
do_tren_may("thời gian một thí nghiệm lớn chia cho một thí nghiệm nhẹ",
            (NGAN_SACH_S / so_lon) / (cong_don / so_nhe), "lần")
do_tren_may("điểm tốt nhất của đội lớn trên tập kiểm định", tot_lon, "%")
do_tren_may("điểm tốt nhất của đội nhẹ trên tập kiểm định", tot_nhe, "%")
print(f"trên máy này, đội {'lớn' if tot_lon > tot_nhe else 'nhẹ' if tot_nhe > tot_lon else 'nào cũng'} "
      f"có điểm tốt nhất cao hơn, chênh {abs(tot_lon - tot_nhe):.2f} điểm")

Nếu tỉ lệ thời gian đo được nhỏ hơn hẳn tỉ lệ số phép tính, phần chênh nằm ở phụ trội cố định (fixed
overhead) của mỗi bước huấn luyện: gọi hàm, cấp phát, cập nhật bộ tối ưu. Phần đó không co lại khi
mô hình nhỏ đi, nên một mô hình nhỏ hơn 100 lần không tự động cho chu kỳ ngắn hơn 100 lần.

Đồ thị trái vẽ điểm tốt nhất tới lúc đó của mỗi đội theo thời gian. Đồ thị phải hỏi một câu khác: nếu
đội nhẹ chỉ được chạy $k$ thí nghiệm rút ngẫu nhiên từ chính những gì nó đã thấy, điểm tốt nhất kỳ
vọng là bao nhiêu? Với $n$ kết quả đã sắp xếp $a_{(1)} \le \dots \le a_{(n)}$, xác suất để thí nghiệm
tốt nhất trong $k$ lần rút là $a_{(i)}$ bằng $(i/n)^k - ((i-1)/n)^k$.

In [ ]:
def tot_nhat_ky_vong(diem, k):
    a = np.sort(np.asarray(diem))
    i = np.arange(1, len(a) + 1)
    return float(np.sum(a * ((i / len(a)) ** k - ((i - 1) / len(a)) ** k)))


diem_nhe = [d for _, d, _ in nhat_ky["nhẹ"]]
k_luoi = np.arange(1, so_nhe + 1)
ky_vong = np.array([tot_nhat_ky_vong(diem_nhe, k) for k in k_luoi])
# Một sự thật toán học, đúng với mọi tập kết quả: mỗi lần rút thêm đem lại ít hơn lần rút trước.
assert np.all(np.diff(ky_vong) >= -1e-9) and np.all(np.diff(ky_vong, 2) <= 1e-9)

DAY = 80
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.4))
for (ten, mau) in [("lớn", MAU[0]), ("nhẹ", MAU[1])]:
    t = np.array([0] + [g for g, _, _ in nhat_ky[ten]])
    d = np.array([d for _, d, _ in nhat_ky[ten]])
    tot = np.maximum.accumulate(d)
    ax1.plot(t[1:], np.clip(d, DAY, None), "o", color=mau, ms=4, alpha=0.45)
    ax1.step(t[1:], tot, where="post", color=mau, lw=2, label=f"đội {ten}: {len(d)} thí nghiệm")
duoi_day = sum(d < DAY for ten in nhat_ky for _, d, _ in nhat_ky[ten])
ax1.text(0.02, 0.97, f"{duoi_day} thí nghiệm dưới {DAY} % vẽ ở mép dưới", transform=ax1.transAxes, fontsize=8, color="0.3", va="top")
ax1.set_ylim(DAY - 1, 100)
ax1.set_xlabel("giây cộng dồn (đo trên máy này)")
ax1.set_ylabel("độ chính xác kiểm định (%)")
ax1.set_title("Điểm tốt nhất tới lúc đó (đo trên máy này)")
ax1.legend(loc="lower right", fontsize=8)

ax2.plot(k_luoi, ky_vong, color=MAU[1], lw=2, label="đội nhẹ: tốt nhất kỳ vọng của k lần rút")
ax2.axhline(tot_lon, color=MAU[0], ls="--", lw=1.2, label="điểm tốt nhất của đội lớn")
for k in sorted({so_lon, so_nhe}):
    ax2.plot([k], [ky_vong[k - 1]], "o", color="0.2", ms=6)
    ax2.annotate(f"k = {k}", (k, ky_vong[k - 1]), xytext=(4, -14), textcoords="offset points", fontsize=8)
ax2.set_ylim(min(ky_vong[0], tot_lon) - 3, max(ky_vong[-1], tot_lon) + 1.5)
ax2.set_xlim(0, 1.15 * so_nhe + 1)
ax2.set_xlabel("số thí nghiệm k")
ax2.set_ylabel("độ chính xác kiểm định (%)")
ax2.set_title("Thêm thí nghiệm, thêm bao nhiêu điểm (đo trên máy này)")
ax2.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()
print(f"đội nhẹ, tốt nhất kỳ vọng với {so_lon} thí nghiệm: {ky_vong[so_lon - 1]:.2f} %, với {so_nhe}: {ky_vong[-1]:.2f} %")

Đường bên phải luôn đi lên và luôn cong xuống, với bất kỳ tập kết quả nào: mỗi thí nghiệm thêm vào
chỉ có thể kéo điểm tốt nhất lên khi nó vượt mọi thí nghiệm trước, và điều đó ngày càng hiếm. Bài tính
của sách thì giả định mỗi vòng thêm một lượng không đổi. Hai cách nhìn này không mâu thuẫn: tìm ngẫu
nhiên là trường hợp xấu nhất, vì mỗi lần rút không học được gì từ lần trước, còn "thí nghiệm hiệu quả"
của sách là thí nghiệm do người thiết kế từ bài học của thí nghiệm trước.

Dù đội nào có điểm cao hơn trên máy của bạn, đồ thị trái cho thấy hai thứ khác nhau. Số thí nghiệm
quyết định một đội tiến được bao xa về phía trần của **chính mô hình nó**. Còn trần đó thì do mô hình
và dữ liệu đầu vào quyết định, ở đây là ảnh 14×14 so với 28×28. Tốc độ lặp mua thêm lần thử, không mua
thêm năng lực của mô hình.

## 4. Ngày 153, thu nhỏ

Chương mở đầu bằng một kịch bản thất bại: ngày 90 mô hình đạt 95 % trên tập test, ngày 120 đạt 96 %
sau một tháng tinh chỉnh kiến trúc, ngày 150 giao cho kỹ sư triển khai, ngày 151 họ báo mô hình cần 4
GB bộ nhớ, ngày 152 có người kiểm đích triển khai: tablet ở phòng khám lưu động chỉ có 512 MB. Ngày
153, năm tháng công sức bị bỏ. Phần ngộ nhận của chương kể một phiên bản khác: một ensemble 2 GB gặp
một đích học máy siêu nhỏ (TinyML) chỉ có 256 KB bộ nhớ.

Ở đây, **giả định** thiết bị của cuộc đua trên là đích TinyML 256 KB ấy, và trọng số lưu ở FP32, 4 byte
mỗi tham số. Sổ tay tính 1 KB = 1,024 byte.

**Dự đoán:** ensemble của đội lớn vượt 256 KB bao nhiêu lần?

In [ ]:
NGAY_95 = sach(90, nguon="ngày 1 / 90 / 120 / 150 / 151 / 152 / 153")
NGAY_96 = sach(120, nguon="ngày 1 / 90 / 120 / 150 / 151 / 152 / 153")
NGAY_BAO = sach(151, nguon="ngày 1 / 90 / 120 / 150 / 151 / 152 / 153")
DC_95 = sach(95, nguon="95 % → 96 %")
DC_96 = sach(96, nguon="95 % → 96 %")
CAN_GB = sach(4, nguon="4 GB vs 512 MB")
CO_MB = sach(512, nguon="4 GB vs 512 MB")
ENSEMBLE_GB = sach(2, nguon="2 GB vs 256 KB")
DICH_KB = sach(256, nguon="2 GB vs 256 KB")
print(f"ngày {NGAY_95} → {NGAY_96}: {NGAY_96 - NGAY_95} ngày tinh chỉnh mua thêm {DC_96 - DC_95} điểm")
print(f"ngày {NGAY_BAO}: cần {CAN_GB} GB = {CAN_GB * 1024:,} MB, gấp {CAN_GB * 1024 / CO_MB:.0f} lần {CO_MB} MB của tablet")
print(f"ví dụ ngộ nhận: {ENSEMBLE_GB} GB = {ENSEMBLE_GB * 1024 ** 2:,} KB, gấp {ENSEMBLE_GB * 1024 ** 2 / DICH_KB:,.0f} "
      f"lần {DICH_KB} KB")

kb_lon = 4 * THANH_VIEN * tham_so_mlp(784, AN_LON) / 1024
kb_nhe = {a: 4 * tham_so_mlp(196, a) / 1024 for a in (16, 32, 64)}
an_toi_da = (DICH_KB * 1024 // 4 - 10) // (784 + 1 + 10)
assert kb_lon > DICH_KB and all(kb <= DICH_KB for kb in kb_nhe.values())
assert 4 * tham_so_mlp(784, an_toi_da) / 1024 <= DICH_KB < 4 * tham_so_mlp(784, an_toi_da + 1) / 1024
print(f"ensemble của đội lớn: {THANH_VIEN} × {tham_so_mlp(784, AN_LON):,} tham số = {kb_lon:,.0f} KB, "
      f"gấp {kb_lon / DICH_KB:.1f} lần đích")
print("mô hình của đội nhẹ: " + ", ".join(f"rộng {a}: {kb:.1f} KB" for a, kb in kb_nhe.items()))
print(f"một MLP 28×28 vừa {DICH_KB} KB khi rộng tối đa {an_toi_da}")

Đội lớn có điểm kiểm định tốt, và không mô hình nào của nó triển khai được. Mọi mô hình của đội nhẹ đều
vừa, vì không gian tìm kiếm của nó đã nằm trong đích. Nếu ràng buộc 256 KB được viết vào bước định
nghĩa bài toán, đội lớn vẫn còn đường đi: tìm trong các MLP 28×28 rộng không quá con số in ở trên, thay
vì bỏ hết kết quả.

Chương gọi đây là nguyên lý lan truyền ràng buộc (constraint propagation), và đưa ra một quy tắc minh
hoạ: ràng buộc phát hiện ở bước thứ $N$ tốn $2^{N-1}$ lần chi phí sửa ở bước 1. Chương nói rõ đây là
một kịch bản minh hoạ độ nhạy, không phải một định luật chi phí đo được.

In [ ]:
BUOC = ["định nghĩa\nbài toán", "thu thập\ndữ liệu", "phát triển\nmô hình", "đánh giá và\nkiểm định",
        "triển khai", "theo dõi và\nbảo trì"]
he_so = [2 ** (n - 1) for n in range(1, len(BUOC) + 1)]
theo_sach("phát hiện ở bước 5", he_so[4], sach=16, nguon="16× ở bước 5, 32× ở bước 6")
theo_sach("phát hiện ở bước 6", he_so[5], sach=32, nguon="32× ở bước 6")
theo_sach("Ví dụ 1.1, ràng buộc lộ ra lúc triển khai", he_so[4], sach=16,
          trich="incurring 16× the cost compared to early correction")

fig, ax = plt.subplots(figsize=(9, 3.8))
mau_cot = [MAU[2]] + ["0.75"] * 3 + [MAU[1], "0.55"]
ax.bar(range(len(BUOC)), he_so, color=mau_cot, width=0.6, edgecolor="white", lw=2)
for i, h in enumerate(he_so):
    ax.text(i, h + 0.6, f"{h}×", ha="center", fontsize=9)
ax.annotate(f"ngày 1: biết {CO_MB} MB từ đầu", (0, 1), xytext=(0.4, 12), fontsize=8,
            arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.annotate(f"ngày {NGAY_BAO}: {CAN_GB} GB gặp {CO_MB} MB\n(sách: 16× ở bước 5)", (4, 16), xytext=(2.3, 24),
            fontsize=8, arrowprops=dict(arrowstyle="-", color="0.5", lw=0.8))
ax.set_xticks(range(len(BUOC)), BUOC, fontsize=8)
ax.set_ylim(0, 37)
ax.set_ylabel("chi phí sửa, so với bước 1")
ax.set_title(r"Quy tắc minh hoạ $2^{N-1}$: phát hiện ràng buộc ở bước $N$")
plt.tight_layout()
plt.show()

## 5. Nhiễu giữa các lần chạy

Chương đòi một phương pháp khoa học cho bước phát triển mô hình: cố định hạt ngẫu nhiên và phiên bản
môi trường, chạy nhiều lần, và so sánh bằng kiểm thử offline theo cặp. Lý do chương nêu là một chênh
lệch độ chính xác 0.5 % có thể quyết định mô hình có đạt ngưỡng lâm sàng hay không. Phần này làm hai
việc. Trước hết, chạy lại cùng một cấu hình với cùng hạt ngẫu nhiên. Sau đó, lấy hai cấu hình tốt nhất
của đội nhẹ và huấn luyện mỗi cấu hình với 5 hạt mới, cặp theo hạt.

**Dự đoán:** hai lần chạy cùng cấu hình, khác hạt ngẫu nhiên, lệch nhau cỡ bao nhiêu điểm?

In [ ]:
CHENH = sach(0.5, trich="a 0.5 percent accuracy difference")
xep = sorted(nhat_ky["nhẹ"], key=lambda r: -r[1])
cfg_a, cfg_b = xep[0][2], (xep[1][2] if len(xep) > 1 else hang_nhe[1])

lan_1 = huan_luyen(X14_hl, y_hl, cfg_a["an"], cfg_a["lr"], cfg_a["batch"], cfg_a["epoch"], 0)
lan_2 = huan_luyen(X14_hl, y_hl, cfg_a["an"], cfg_a["lr"], cfg_a["batch"], cfg_a["epoch"], 0)
lech_max = max((p - q).abs().max().item() for p, q in zip(lan_1.parameters(), lan_2.parameters()))
print(f"cùng cấu hình, cùng hạt: trọng số lệch nhau tối đa {lech_max:g}"
      f"{' (giống hệt)' if lech_max == 0 else ''}")

HAT = [1, 2, 3, 4, 5]
diem = np.array([[thi_nghiem_nhe(c, hat=h) for h in HAT] for c in (cfg_a, cfg_b)])
hieu = diem[0] - diem[1]
for ten, c, d in [("A", cfg_a, diem[0]), ("B", cfg_b, diem[1])]:
    print(f"cấu hình {ten}: rộng {c['an']}, {c['epoch']} epoch, batch {c['batch']}, tốc độ học {c['lr']:.4f}")
    do_tren_may(f"cấu hình {ten}, độ lệch chuẩn qua {len(HAT)} hạt", d.std(ddof=1), "điểm")
    do_tren_may(f"cấu hình {ten}, khoảng cao nhất trừ thấp nhất", np.ptp(d), "điểm")
do_tren_may("A trừ B, trung bình qua các cặp", hieu.mean(), "điểm")
do_tren_may("A trừ B, độ lệch chuẩn qua các cặp", hieu.std(ddof=1), "điểm")
do_tren_may("thống kê t của kiểm thử theo cặp", hieu.mean() / (hieu.std(ddof=1) / math.sqrt(len(HAT)) + 1e-12))

# Riêng tập kiểm định đã có nhiễu: sai số chuẩn của một tỉ lệ đo trên n ảnh là sqrt(p(1−p)/n).
p = 0.9  # giả định: độ chính xác quanh 90 %
se = 100 * math.sqrt(p * (1 - p) / N_KD)
n_can = math.ceil(4 * p * (1 - p) / (CHENH / 100) ** 2)
print(f"ở {100 * p:.0f} % trên {N_KD:,} ảnh kiểm định, sai số chuẩn là {se:.2f} điểm, lớn hơn {CHENH} điểm")
print(f"để hai sai số chuẩn còn {CHENH} điểm, cần {n_can:,} ảnh")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for j, h in enumerate(HAT):
    ax1.plot([0, 1], diem[:, j], "-", color="0.75", lw=1)
ax1.plot(np.zeros(len(HAT)), diem[0], "o", color=MAU[1], ms=7)
ax1.plot(np.ones(len(HAT)), diem[1], "o", color=MAU[4], ms=7)
ax1.set_xticks([0, 1], ["cấu hình A", "cấu hình B"])
ax1.set_xlim(-0.5, 1.5)
ax1.set_ylabel("độ chính xác kiểm định (%)")
ax1.set_title("Cùng hạt nối bằng một đường (đo trên máy này)")

ax2.bar(range(len(HAT)), hieu, color=[MAU[1] if v > 0 else MAU[4] for v in hieu], width=0.6)
ax2.axhline(0, color="0.3", lw=1)
for v in (CHENH, -CHENH):
    ax2.axhline(v, color="0.4", ls="--", lw=1)
ax2.text(len(HAT) - 0.6, CHENH, f"±{CHENH} điểm (chênh lệch chương nêu)", fontsize=8, va="bottom", ha="right", color="0.3")
ax2.set_xticks(range(len(HAT)), [f"hạt {h}" for h in HAT])
lim = max(1.0, 1.3 * np.abs(hieu).max())
ax2.set_ylim(-lim, lim)
ax2.set_ylabel("A trừ B (điểm)")
ax2.set_title("Chênh lệch theo từng cặp (đo trên máy này)")
plt.tight_layout()
plt.show()

Cùng hạt ngẫu nhiên, cùng phiên bản thư viện, cùng máy thì cho cùng trọng số: đó là phần tính tái lập
(reproducibility) mà một hệ thống theo dõi thí nghiệm (experiment tracking) phải ghi lại được. Đổi hạt
thì kết quả đổi, và đọc độ lệch chuẩn ở trên để thấy đổi bao nhiêu. Nếu nó cùng cỡ với 0.5 điểm, một
lần chạy cho mỗi cấu hình không đủ để nói cấu hình nào tốt hơn. Kiểm thử theo cặp giúp vì nó trừ đi phần
nhiễu hai cấu hình chia nhau ở cùng một hạt; nếu các cột bên phải đổi dấu giữa các hạt, chênh lệch
trung bình chưa phân biệt được với nhiễu.

Còn một cái bẫy nằm ngay trong cuộc đua ở phần 3. Cấu hình A đứng đầu sau đúng một lần chạy, với hạt
0. Chọn lần chạy tốt nhất trong nhiều lần chạy đơn lẻ là chọn cả phần may của nó. Nếu A không còn hơn
B khi chạy lại với hạt mới, bạn đang thấy chính hiện tượng đó.

Phần cuối không phụ thuộc máy: trên 2,000 ảnh kiểm định, chỉ riêng việc đếm đúng sai đã có sai số
chuẩn khoảng 0.67 điểm ở độ chính xác 90 %. Một chênh lệch 0.5 % nhỏ hơn một sai số chuẩn của phép đo.
Với hệ thống sàng lọc của chương, mẫu số của độ nhạy chỉ là số ca có bệnh, còn ít hơn tổng số ảnh, nên
sai số chuẩn còn lớn hơn.

## Thử thêm

1. Ở phần 1, đặt `TRAN = 100` (bỏ trần) rồi chạy lại hai phần đầu. Mô hình nhẹ thắng bao nhiêu điểm, và
   cần ít nhất bao nhiêu thí nghiệm hiệu quả?
2. Ở phần 3, cho đội nhẹ dùng ảnh 28×28: trong `thi_nghiem_nhe`, đổi `X14_hl`, `X14_kd` thành `X28_hl`,
   `X28_kd`, và chỉ rút độ rộng trong 16, 32, 64, 82, để mọi mô hình vẫn vừa 256 KB. Đội nhẹ còn chạy
   được bao nhiêu thí nghiệm, và điểm tốt nhất của nó đổi thế nào so với đội lớn?
3. Ở phần 5, đổi `HAT` thành 10 hạt. Thống kê t có ổn định hơn không? Tính lại `n_can` với `p = 0.97`:
   ở độ chính xác cao, cần ít hay nhiều ảnh hơn để thấy chênh lệch 0.5 điểm?

## Tóm lại

* Theo bài tính của chương, sau 26 tuần mô hình lớn đạt 98.9 %, còn mô hình nhẹ đạt 100 % trước trần
  và 99 % sau trần, hơn đúng 0.1 điểm, tức một vòng của chính nó.
* Kết luận nằm sát mép: 89 thay vì 100 thí nghiệm hiệu quả là đủ đổi chiều. Trần chặn phần thắng ở 0.1
  điểm, còn phần thua có thể tới 8.9 điểm.
* Trong một cuộc tìm siêu tham số thật, điểm tốt nhất kỳ vọng tăng chậm dần theo số thí nghiệm. Tốc
  độ lặp mua thêm lần thử, không mua thêm năng lực của mô hình.
* Ràng buộc 256 KB biết từ ngày đầu thu hẹp không gian tìm kiếm. Biết lúc triển khai, theo quy tắc
  minh hoạ của chương, giá gấp 16 lần, lúc theo dõi gấp 32 lần.
* Trên 2,000 ảnh kiểm định, sai số chuẩn ở 90 % đã khoảng 0.67 điểm. Một chênh lệch 0.5 % cần nhiều lần
  chạy, kiểm thử theo cặp và tập kiểm định đủ lớn mới tách được khỏi nhiễu.

---
### Nguồn và giấy phép

Sổ tay này đi kèm bài học [Quy trình hệ thống học máy](https://d3lu8vk4we0dc.cloudfront.net/lessons/vol1-ch03-ml-workflow/), dựng từ chương
[*ML Workflow*](https://mlsysbook.ai/vol1/ml_workflow/ml_workflow.html) của sách. Một phần sổ tay phỏng theo phòng thí nghiệm `labs/vol1/lab_03_ml_workflow.py` của sách (© 2026 President and Fellows of Harvard College, CC BY-NC-SA 4.0).

Nội dung gốc thuộc Machine Learning Systems của Vijay Janapa Reddi và cộng sự, MIT Press. Bản quyền © 2024-2026 Harvard University. Giấy phép CC BY-NC-SA 4.0. Bản tiếng Việt này là tác phẩm phái sinh dùng cùng giấy phép: ghi công, chia sẻ tương tự, phi thương mại.